In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# -----------------------
# Paths
# -----------------------
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

csv_path = PROJECT_ROOT / "data" / "processed" / "ims_features_sample.csv"
assert csv_path.exists(), f"Missing features file: {csv_path}"

df = pd.read_csv(csv_path)
print("Loaded:", df.shape)
print("Label counts:\n", df["label"].value_counts())

# -----------------------
# 1) Prevent leakage: split by file
# -----------------------
# NOTE: This avoids train/val sharing windows from the same file.
files = df["file_name"].unique()

# Shuffle files for a fair split (still reproducible)
rng = np.random.default_rng(42)
files = rng.permutation(files)

split = int(0.75 * len(files))
train_files = set(files[:split])
val_files = set(files[split:])

train_df = df[df["file_name"].isin(train_files)].copy()
val_df   = df[df["file_name"].isin(val_files)].copy()

print("\nTrain rows:", train_df.shape, "| Val rows:", val_df.shape)
print("Train label counts:\n", train_df["label"].value_counts())
print("Val label counts:\n", val_df["label"].value_counts())

# -----------------------
# 2) Remove leakage / metadata columns
# -----------------------
# Remove anything that encodes ordering/time/file identity.
DROP_COLS = [
    "test_dir",
    "file_name",
    "label",
    "file_index",     # remove if present
    "window_index",   # remove if present
]

# Only drop columns that exist
DROP_COLS = [c for c in DROP_COLS if c in train_df.columns]

X_train = train_df.drop(columns=DROP_COLS)
y_train = train_df["label"].astype(int)

X_val = val_df.drop(columns=DROP_COLS)
y_val = val_df["label"].astype(int)

print("\nX_train shape:", X_train.shape, "X_val shape:", X_val.shape)

# -----------------------
# 3) Clean inf/NaN (safe)
# -----------------------
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val   = X_val.replace([np.inf, -np.inf], np.nan)

# -----------------------
# 4) Models + Evaluation helper
# -----------------------
def evaluate_model(name, y_true, y_pred, y_proba):
    print(f"\n=== {name} ===")
    print("AUC:", roc_auc_score(y_true, y_proba))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=4))


# -----------------------
# Logistic Regression (pipeline = best practice)
# -----------------------
lr = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000))
])

lr.fit(X_train, y_train)
pred_lr = lr.predict(X_val)
proba_lr = lr.predict_proba(X_val)[:, 1]
evaluate_model("Logistic Regression (leakage-safe)", y_val, pred_lr, proba_lr)

# -----------------------
# Random Forest (impute separately)
# -----------------------
# For RF we can impute median and fill leftovers with 0
X_train_rf = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X_train), columns=X_train.columns)
X_val_rf   = pd.DataFrame(SimpleImputer(strategy="median").fit(X_train).transform(X_val), columns=X_val.columns)

# If any column was entirely NaN, median imputer can still leave NaN → fill 0
X_train_rf = X_train_rf.fillna(0)
X_val_rf   = X_val_rf.fillna(0)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train_rf, y_train)
pred_rf = rf.predict(X_val_rf)
proba_rf = rf.predict_proba(X_val_rf)[:, 1]
evaluate_model("Random Forest (leakage-safe)", y_val, pred_rf, proba_rf)

# -----------------------
# Optional: XGBoost (if installed)
# -----------------------
try:
    from xgboost import XGBClassifier

    xgb = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )

    # Use same imputed matrices as RF
    xgb.fit(X_train_rf, y_train)
    pred_xgb = xgb.predict(X_val_rf)
    proba_xgb = xgb.predict_proba(X_val_rf)[:, 1]
    evaluate_model("XGBoost (leakage-safe)", y_val, pred_xgb, proba_xgb)

except Exception as e:
    print("\n[XGBoost skipped] Install with: pip install xgboost")
    print("Reason:", e)

Loaded: (2340, 46)
label
0    1989
1     351
Name: count, dtype: int64
NaN columns (top):
ch7_rms         1560
ch7_p2p         1560
ch7_kurtosis    1560
ch8_mean        1560
ch5_mean        1560
ch5_rms         1560
ch5_std         1560
ch6_mean        1560
ch5_kurtosis    1560
ch6_std         1560
dtype: int64
Total NaNs: 31200
Total NaNs after fix: 0

=== Logistic Regression ===
AUC: 1.0
[[497   0]
 [  0  88]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       497
           1       1.00      1.00      1.00        88

    accuracy                           1.00       585
   macro avg       1.00      1.00      1.00       585
weighted avg       1.00      1.00      1.00       585


=== Random Forest ===
AUC: 1.0
[[497   0]
 [  0  88]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       497
           1       1.00      1.00      1.00        88

    accuracy                          

file_index      0.814922
ch3_p2p         0.007732
ch1_rms         0.007438
ch4_p2p         0.006801
ch2_kurtosis    0.006761
ch1_mean        0.006505
ch1_p2p         0.006383
ch3_kurtosis    0.006318
ch4_mean        0.006225
ch1_std         0.006109
ch2_rms         0.006015
ch4_rms         0.005914
ch1_kurtosis    0.005750
ch2_std         0.005749
ch4_std         0.005571
dtype: float64